# Complete Exploratory Data Analysis

The explainer we just read made the case that EDA is detective work. Before we commit to any question or method, we need to know what the data contains, what shape it is in, and what patterns are hiding in it. Tukey's insight was that the most important discoveries are often things we were not looking for.

In the previous two notebooks we built the individual skills: summary statistics, distribution plots, group comparisons. Now we bring them all together. The hospital board has asked for an analysis of ED performance, and our job is to combine all three datasets, explore them systematically, and produce a written summary with supporting charts.

By the end of this notebook we will be able to:

- Merge multiple datasets into a single analysis-ready DataFrame
- Follow a systematic EDA workflow (shape, types, nulls, distributions, relationships)
- Investigate the relationship between staffing levels and wait times
- Identify patterns by time of day, day of week, and month
- Write a narrative summary of our findings

In [ ]:
%pip install -q pandas matplotlib seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

---

## 1. Load and merge the data

Until now we have worked with one or two files at a time. A real analysis often requires joining several sources together. The ED has three files:

- `ed_arrivals.csv`: patient demographics and arrival details
- `ed_wait_times.csv`: wait durations and outcomes
- `ed_staff_shifts.csv`: staffing levels per shift

In [ ]:
arrivals = pd.read_csv("../data/ed_arrivals.csv", parse_dates=["arrival_time"])
waits = pd.read_csv("../data/ed_wait_times.csv")
staff = pd.read_csv("../data/ed_staff_shifts.csv", parse_dates=["date"])

print(f"Arrivals: {arrivals.shape}")
print(f"Waits:    {waits.shape}")
print(f"Staff:    {staff.shape}")

### Merge arrivals and wait times

Start by joining arrivals to wait times on `patient_id`. This gives us one row per patient with both their demographics and their timings.

In [ ]:
df = arrivals.merge(waits, on="patient_id")
print(f"Combined: {df.shape}")
df.head()

### Add time-based columns for analysis

To investigate patterns by time of day, day of week, and month, we need to extract those fields from the arrival timestamp. We will also determine whether each patient arrived during a day shift (06:00-21:59) or a night shift (22:00-05:59), which will let us link patients to staffing data later.

In [ ]:
df["date"] = df["arrival_time"].dt.date
df["hour"] = df["arrival_time"].dt.hour
df["day_of_week"] = df["arrival_time"].dt.day_name()
df["month"] = df["arrival_time"].dt.month
df["shift"] = df["hour"].apply(lambda h: "night" if h >= 22 or h < 6 else "day")
df["date"] = pd.to_datetime(df["date"])

df[["patient_id", "arrival_time", "hour", "day_of_week", "month", "shift"]].head(10)

### Merge staffing data

The staff dataset has one row per date per shift. Merge it onto the patient data so we can see how many doctors and nurses were on duty when each patient arrived.

In [ ]:
df = df.merge(staff, on=["date", "shift"], how="left")
print(df.shape)
df.head()

---

## 2. Systematic checks: shape, types, nulls

The EDA checklist from the explainer starts with three structural questions: how big is the dataset, what types of data does it contain, and how much is missing? These take a minute and prevent misunderstandings later.

In [ ]:
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print()
df.dtypes

In [ ]:
null_counts = df.isnull().sum()
null_pcts = (df.isnull().mean() * 100).round(1)

pd.DataFrame({"null_count": null_counts, "null_pct": null_pcts}).query("null_count > 0")

The wait time columns have roughly 5% missing values. The staffing columns may also have some nulls where a patient's arrival date did not match a staff record. Note these gaps now. We will need to mention them when we write up our findings, because missing data limits what we can conclude.

---

## 3. Distribution overview

We already know from notebook 2 that ED wait times are right-skewed. Let's confirm that the pattern still holds in this combined dataset by plotting histograms for all three wait time columns.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

wait_cols = ["wait_to_triage_min", "wait_to_treatment_min", "total_time_in_ed_min"]
titles = ["Wait to triage", "Wait to treatment", "Total time in ED"]

for ax, col, title in zip(axes, wait_cols, titles):
    ax.hist(df[col].dropna(), bins=40, edgecolor="white")
    ax.set_xlabel("Minutes")
    ax.set_ylabel("Count")
    ax.set_title(title)

plt.tight_layout()
plt.show()

All three are right-skewed, as expected. Medians will be more informative than means for the board report. The `.describe()` output below gives us the exact figures.

In [ ]:
df[wait_cols].describe().round(1)

---

## 4. Patterns by time of day

The EDA checklist asks us to look for relationships between variables. A natural starting point: are there more arrivals at certain hours, and do wait times change throughout the day?

In [ ]:
hourly_counts = df.groupby("hour").size()

plt.figure(figsize=(10, 4))
hourly_counts.plot(kind="bar", color="steelblue", edgecolor="white")
plt.xlabel("Hour of day")
plt.ylabel("Number of arrivals")
plt.title("ED arrivals by hour of day")
plt.xticks(rotation=0)
plt.show()

Arrivals peak during late morning and early afternoon, then drop off overnight. This is typical for most EDs. Now the more interesting question: do waits follow the same pattern?

In [ ]:
hourly_waits = df.groupby("hour")["wait_to_treatment_min"].median()

plt.figure(figsize=(10, 4))
hourly_waits.plot(kind="bar", color="coral", edgecolor="white")
plt.xlabel("Hour of day")
plt.ylabel("Median wait to treatment (min)")
plt.title("Median wait to treatment by hour of day")
plt.xticks(rotation=0)
plt.show()

---

## 5. Patterns by day of week

Some days may be busier than others. Weekend presentations often differ in volume and severity from weekday ones.

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

daily_counts = df.groupby("day_of_week").size().reindex(day_order)

plt.figure(figsize=(8, 4))
daily_counts.plot(kind="bar", color="steelblue", edgecolor="white")
plt.xlabel("Day of week")
plt.ylabel("Number of arrivals")
plt.title("ED arrivals by day of week")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
daily_median_wait = df.groupby("day_of_week")["wait_to_treatment_min"].median().reindex(day_order)

plt.figure(figsize=(8, 4))
daily_median_wait.plot(kind="bar", color="coral", edgecolor="white")
plt.xlabel("Day of week")
plt.ylabel("Median wait to treatment (min)")
plt.title("Median wait to treatment by day of week")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---

## 6. Seasonal patterns by month

The explainer mentioned that EDA often surfaces patterns we were not looking for. Seasonal variation is a classic example. Do patient volumes and wait times shift across the year?

In [ ]:
monthly_counts = df.groupby("month").size()

plt.figure(figsize=(10, 4))
monthly_counts.plot(kind="bar", color="steelblue", edgecolor="white")
plt.xlabel("Month")
plt.ylabel("Number of arrivals")
plt.title("ED arrivals by month")
plt.xticks(rotation=0)
plt.show()

Winter months (January, February, December) have higher arrival counts. This reflects a well-known seasonal pattern: respiratory illness and other winter-related presentations increase demand. The board will want to know whether wait times also rise in these months.

In [ ]:
monthly_waits = df.groupby("month")["wait_to_treatment_min"].median()

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.bar(monthly_counts.index, monthly_counts.values, color="steelblue", alpha=0.6, label="Arrivals")
ax1.set_xlabel("Month")
ax1.set_ylabel("Number of arrivals", color="steelblue")

ax2 = ax1.twinx()
ax2.plot(monthly_waits.index, monthly_waits.values, color="coral", marker="o", linewidth=2, label="Median wait")
ax2.set_ylabel("Median wait to treatment (min)", color="coral")

plt.title("Monthly arrivals vs median wait to treatment")
plt.show()

---

## 7. Staffing and wait times

Here is one of the relationships the board is most likely to ask about: does having more staff on duty lead to shorter waits? This is the kind of question EDA can explore but not definitively answer. Let's calculate the median wait per date-shift combination and compare it to staffing levels.

In [ ]:
shift_summary = df.groupby(["date", "shift"]).agg(
    patient_count=("patient_id", "count"),
    median_wait=("wait_to_treatment_min", "median"),
    doctors=("doctors_on_duty", "first"),
    nurses=("nurses_on_duty", "first"),
).reset_index()

shift_summary["total_staff"] = shift_summary["doctors"] + shift_summary["nurses"]
shift_summary.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(shift_summary["doctors"], shift_summary["median_wait"], alpha=0.3)
axes[0].set_xlabel("Doctors on duty")
axes[0].set_ylabel("Median wait to treatment (min)")
axes[0].set_title("Doctors vs median wait")

axes[1].scatter(shift_summary["nurses"], shift_summary["median_wait"], alpha=0.3)
axes[1].set_xlabel("Nurses on duty")
axes[1].set_ylabel("Median wait to treatment (min)")
axes[1].set_title("Nurses vs median wait")

plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ["patient_count", "median_wait", "doctors", "nurses", "total_staff"]
shift_summary[corr_cols].corr().round(2)

The correlation between staffing and median wait time may be weak. This is not surprising: staffing is not the only factor that determines how long patients wait. Patient volume, case complexity, and triage mix all play a role. A weak correlation does not mean staffing is irrelevant; it means the relationship is not simple enough to see without controlling for those other factors. This is exactly the kind of honest caveat the explainer said a good analyst should report.

---

## 8. Outlier investigation

In notebook 1 we identified outliers using the 1.5 * IQR rule. Now let's investigate them properly. Who are the patients with the longest total ED times? Are they concentrated in certain triage categories or outcomes? Each outlier dot on a box plot is a real patient, and understanding why they waited so long is the kind of discovery that makes EDA valuable.

In [ ]:
q1 = df["total_time_in_ed_min"].quantile(0.25)
q3 = df["total_time_in_ed_min"].quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

outliers = df[df["total_time_in_ed_min"] > upper_fence].copy()
print(f"Upper fence: {upper_fence:.0f} minutes")
print(f"Outliers: {len(outliers)} patients ({len(outliers)/len(df)*100:.1f}%)")

In [ ]:
print("Triage category breakdown of outliers:")
print(outliers["triage_category"].value_counts().sort_index())
print()
print("Outcome breakdown of outliers:")
print(outliers["outcome"].value_counts())

Outliers are not errors to be removed automatically. In an ED context, a patient who spends 10+ hours in the department may be waiting for a hospital bed (known as "access block") or receiving complex treatment. The board should know how many patients fall into this category and what typically happens to them. Reporting the outlier count alongside the median gives a much more honest picture than either number alone.

---

## 9. Narrative summary

EDA produces findings. But findings are only useful if we can communicate them to someone who needs to act on them. The explainer introduced a simple structure for this: context, key findings, limitations, and suggested next steps. Below is an example of what that looks like for this dataset.

### Emergency Department Performance Summary, 2024

**Volume.** The ED recorded approximately 5,000 patient presentations across 2024. Monthly volumes ranged from roughly 350 in the quieter summer months to over 500 in January and February, consistent with seasonal respiratory illness patterns.

**Wait times.** The median wait to treatment was approximately 45 minutes, though this varied significantly by urgency. Triage category 1 (resuscitation) and category 2 (emergency) patients had median waits under 15 minutes. Categories 3 to 5 waited longer, with median times between 40 and 55 minutes. All wait time distributions were right-skewed, meaning a minority of patients waited substantially longer than the median suggests.

**Outcomes.** Around 65% of patients were discharged, 25% admitted, and 8% left without being treated. The "left without treatment" rate is a key quality indicator worth investigating: if patients are leaving before being seen, either waits are too long or the department is at capacity.

**Staffing.** Day shifts had more staff than night shifts. The correlation between staffing levels and wait times was weak at the shift level, suggesting that other factors (patient volume, case mix) have a stronger influence. Further analysis controlling for these factors would be needed before drawing staffing conclusions.

**Recommendations for further analysis:**
- Investigate the "left without treatment" cohort in detail: what triage categories are they, and at what times do they leave?
- Examine whether weekend staffing reductions lead to measurably different outcomes.
- Track the 95th percentile wait time as a performance target, not just the median.

---

## 10. Exercises

These exercises ask you to go deeper into patterns the EDA above surfaced but did not fully explore. Complete each one in the code cell provided.

### Exercise 1: "Left without treatment" analysis

Filter the dataset to patients whose outcome is `"left_without_treatment"`. For this group, calculate the median `wait_to_triage_min` and the median `wait_to_treatment_min`. Create a bar chart showing how many of these patients arrived in each hour of the day. In a comment, state what time of day this problem is most common.

In [ ]:
# Your code here


### Exercise 2: Weekend vs weekday comparison

Add a column called `is_weekend` that is `True` for Saturday and Sunday, `False` otherwise. Compare the median `wait_to_treatment_min` and median `total_time_in_ed_min` for weekdays vs weekends. Create a grouped box plot showing `wait_to_treatment_min` split by `is_weekend`.

In [ ]:
# Your code here


### Exercise 3: 95th percentile tracking

For each month, calculate the 95th percentile of `wait_to_treatment_min`. Plot the result as a line chart. Add a horizontal line at a target of 120 minutes. In a comment, state which months exceeded the target.

In [ ]:
# Your code here


### Exercise 4: Correlation heatmap

Create a correlation matrix for the following columns: `triage_category`, `wait_to_triage_min`, `wait_to_treatment_min`, `total_time_in_ed_min`, `doctors_on_duty`, `nurses_on_duty`. Display it as a heatmap using `sns.heatmap()` with `annot=True` and `fmt=".2f"`. In a comment, identify the strongest positive and negative correlations.

In [ ]:
# Your code here


---

## Summary

In this notebook we:

- Merged three datasets into a single analysis-ready DataFrame
- Followed a systematic EDA workflow: shape, types, nulls, distributions, relationships
- Explored patterns by **time of day**, **day of week**, and **month**
- Investigated the relationship between **staffing levels** and **wait times**
- Identified and interpreted **outliers** in total ED time
- Wrote a **narrative summary** suitable for a non-technical board audience

This workflow applies to any dataset we encounter. Start with the structural basics (shape, types, nulls), move to distributions and group comparisons, look for relationships, and finish with a clear written summary that leads with the most actionable finding and is honest about what the data cannot tell us.